In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install optuna xgboost lightgbm "mlflow<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.7/774.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: cachetools
    Foun

In [1]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(f"{base_folder}/data/airline.db")

airline = pd.read_sql_query(
    """
    SELECT
        p.passenger_id,
        p.gender,
        p.customer_type,
        p.travel_type,
        p.class,
        f.flight_distance,
        f.departure_delay,
        f.arrival_delay,
        sr.inflight_wifi,
        sr.time_convenient,
        sr.online_booking,
        sr.gate_location,
        sr.food_drink,
        sr.online_boarding,
        sr.seat_comfort,
        sr.inflight_entertainment,
        sr.onboard_service,
        sr.legroom,
        sr.baggage,
        sr.checkin,
        sr.inflight_service,
        sr.cleanliness,
        sr.satisfaction
    FROM passenger AS p
    JOIN service_ratings AS sr
        ON sr.passenger_id = p.passenger_id
    JOIN flight AS f
        ON f.flight_id = sr.flight_id
    ORDER BY p.passenger_id
    """,
    conn,
)

conn.close()

airline.head()

,passenger_id,gender,customer_type,travel_type,class,flight_distance,departure_delay,arrival_delay,inflight_wifi,time_convenient,...,online_boarding,seat_comfort,inflight_entertainment,onboard_service,legroom,baggage,checkin,inflight_service,cleanliness,satisfaction
0,1,Male,Loyal Customer,Personal Travel,Eco Plus,460,25,18.0,3,4,...,3,5,5,4,3,4,4,5,5,neutral or dissatisfied
1,2,Male,disloyal Customer,Business travel,Business,235,1,6.0,3,2,...,3,1,1,1,5,3,1,4,1,neutral or dissatisfied
2,3,Female,Loyal Customer,Business travel,Business,1142,0,0.0,2,2,...,5,5,5,4,3,4,4,4,5,satisfied
3,4,Female,Loyal Customer,Business travel,Business,562,11,9.0,2,5,...,2,2,2,2,5,3,1,4,2,neutral or dissatisfied
4,5,Male,Loyal Customer,Business travel,Business,214,0,0.0,3,3,...,5,5,3,3,4,4,3,3,3,satisfied


In [3]:
# =============================================================================
# FULL PIPELINE with OPTUNA for AIRLINE SATISFACTION CLASSIFICATION
# - Build preprocessing
# - Stratified train/test split
# - Train & log 4 models WITHOUT PCA (RidgeClassifier, HGB, XGBoost, LightGBM)
# - Train & log 4 models WITH PCA
# - Pick GLOBAL best among 8 models by Test F1
# - Save the global best model
# =============================================================================

import time
import os
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, make_scorer
from sklearn.base import clone
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import mlflow
from mlflow.models import infer_signature
import joblib

import optuna
from optuna.samplers import TPESampler

# Import Shared components
from airline_pipeline import build_preprocessing, make_estimator_for_name

start_time = time.monotonic()
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =============================================================================
# STEP 1: Build Preprocessing Pipeline
# =============================================================================
preprocessing = build_preprocessing()
print("✓ STEP 1: Preprocessing pipeline created.")

# =============================================================================
# STEP 2: Stratified Train/Test Split
# =============================================================================
airline = airline.dropna()
X = airline.drop(["satisfaction", "passenger_id"], axis=1).copy()
y = airline["satisfaction"].copy()

# Convert target to binary 0/1
y_enc = (y == "satisfied").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)
print(f"✓ STEP 2: Stratified split done. Train size: {len(X_train)}, Test size: {len(X_test)}")

# F1 scorer for Optuna
f1_scorer = make_scorer(f1_score, average='binary', pos_label=1)

# =============================================================================
# STEP 3: Configure MLflow
# =============================================================================
load_dotenv(
    dotenv_path="/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/notebooks/.env",
    override=True
)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("airline_satisfaction_optuna")
print("✓ STEP 3: MLflow configured.")

# =============================================================================
# STEP 4: Define Optuna objective functions
# =============================================================================
def objective_ridge(trial, preprocessing, X_train, y_train):
    alpha = trial.suggest_float("ridge__alpha", 0.1, 100.0, log=True)
    pipeline = make_pipeline(clone(preprocessing), RidgeClassifier(alpha=alpha))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring=f1_scorer, n_jobs=-1)
    return 1 - scores.mean()  # minimize 1-F1

def objective_hgb(trial, preprocessing, X_train, y_train):
    learning_rate = trial.suggest_float("hgb__learning_rate", 0.05, 0.2)
    max_depth = trial.suggest_int("hgb__max_depth", 3, 8)
    pipeline = make_pipeline(clone(preprocessing),
                             HistGradientBoostingClassifier(learning_rate=learning_rate, max_depth=max_depth, random_state=42))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring=f1_scorer, n_jobs=-1)
    return 1 - scores.mean()

def objective_xgb(trial, preprocessing, X_train, y_train):
    learning_rate = trial.suggest_float("xgb__learning_rate", 0.05, 0.2)
    max_depth = trial.suggest_int("xgb__max_depth", 3, 8)
    n_estimators = trial.suggest_int("xgb__n_estimators", 100, 300, step=50)
    pipeline = make_pipeline(clone(preprocessing),
                             XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate,
                                           max_depth=max_depth, use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring=f1_scorer, n_jobs=-1)
    return 1 - scores.mean()

def objective_lgbm(trial, preprocessing, X_train, y_train):
    learning_rate = trial.suggest_float("lgbm__learning_rate", 0.05, 0.2)
    num_leaves = trial.suggest_int("lgbm__num_leaves", 20, 80)
    n_estimators = trial.suggest_int("lgbm__n_estimators", 100, 300, step=50)
    pipeline = make_pipeline(clone(preprocessing),
                             LGBMClassifier(n_estimators=n_estimators,
                                            learning_rate=learning_rate,
                                            num_leaves=num_leaves, random_state=42, n_jobs=-1))
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring=f1_scorer, n_jobs=-1)
    return 1 - scores.mean()

# =============================================================================
# STEP 5: Run Optuna studies (NO PCA)
# =============================================================================
model_names = ["ridge", "histgradientboosting", "xgboost", "lightgbm"]
objective_functions = {"ridge": objective_ridge, "histgradientboosting": objective_hgb, "xgboost": objective_xgb, "lightgbm": objective_lgbm}
results = {}

for name in model_names:
    print(f"\n{'='*80}\nOptimizing {name.upper()} (NO PCA) - 10 trials\n{'='*80}")
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
    study.optimize(lambda trial: objective_functions[name](trial, preprocessing, X_train, y_train), n_trials=10, show_progress_bar=True)
    best_params = study.best_params

    # Build final pipeline
    if name=="ridge":
      final_model = make_pipeline(clone(preprocessing), RidgeClassifier(alpha=best_params["ridge__alpha"]))
    elif name=="histgradientboosting":
      final_model = make_pipeline(clone(preprocessing), HistGradientBoostingClassifier(learning_rate=best_params["hgb__learning_rate"], max_depth=best_params["hgb__max_depth"], random_state=42))
    elif name=="xgboost":
      final_model = make_pipeline(clone(preprocessing), XGBClassifier(n_estimators=best_params["xgb__n_estimators"], learning_rate=best_params["xgb__learning_rate"], max_depth=best_params["xgb__max_depth"], use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1))
    elif name=="lightgbm":
      final_model = make_pipeline(clone(preprocessing), LGBMClassifier(n_estimators=best_params["lgbm__n_estimators"], learning_rate=best_params["lgbm__learning_rate"], num_leaves=best_params["lgbm__num_leaves"], random_state=42, n_jobs=-1))

    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)
    test_f1 = f1_score(y_test, y_pred)*100

    cv_f1 = (1 - study.best_value)*100
    print(f"{name} (no PCA) CV F1: {cv_f1:.2f}% | Test F1: {test_f1:.2f}%")
    results[name] = {"pipeline": final_model, "test_f1": test_f1, "cv_f1": cv_f1}

    # MLflow log
    with mlflow.start_run(run_name=f"{name}_baseline_optuna"):
        mlflow.log_param("model_family", name)
        mlflow.log_param("uses_pca", False)
        mlflow.log_params(best_params)
        mlflow.log_metric("cv_F1", cv_f1)
        mlflow.log_metric("test_F1", test_f1)
        signature = infer_signature(X_train, final_model.predict(X_train))
        mlflow.sklearn.log_model(final_model, artifact_path="airline_model", signature=signature, input_example=X_train)

# =============================================================================
# STEP 6: PCA models
# =============================================================================
pca_results = {}
def create_pca_pipeline(base_name, best_params):
    if base_name=="ridge": return make_pipeline(clone(preprocessing), PCA(n_components=best_params["pca__n_components"]), RidgeClassifier(alpha=best_params["ridge__alpha"]))
    elif base_name=="histgradientboosting": return make_pipeline(clone(preprocessing), PCA(n_components=best_params["pca__n_components"]), HistGradientBoostingClassifier(learning_rate=best_params["hgb__learning_rate"], max_depth=best_params["hgb__max_depth"], random_state=42))
    elif base_name=="xgboost": return make_pipeline(clone(preprocessing), PCA(n_components=best_params["pca__n_components"]), XGBClassifier(n_estimators=best_params["xgb__n_estimators"], learning_rate=best_params["xgb__learning_rate"], max_depth=best_params["xgb__max_depth"], use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1))
    elif base_name=="lightgbm": return make_pipeline(clone(preprocessing), PCA(n_components=best_params["pca__n_components"]), LGBMClassifier(n_estimators=best_params["lgbm__n_estimators"], learning_rate=best_params["lgbm__learning_rate"], num_leaves=best_params["lgbm__num_leaves"], random_state=42, n_jobs=-1))

pca_objectives = {
    "ridge_with_pca": lambda trial: objective_ridge(
        trial,
        make_pipeline(clone(preprocessing), PCA(n_components=trial.suggest_float("pca__n_components", 0.90, 0.99))),
        X_train,
        y_train
    ),
    "histgradientboosting_with_pca": lambda trial: objective_hgb(
        trial,
        make_pipeline(clone(preprocessing), PCA(n_components=trial.suggest_float("pca__n_components", 0.90, 0.99))),
        X_train,
        y_train
    ),
    "xgboost_with_pca": lambda trial: objective_xgb(
        trial,
        make_pipeline(clone(preprocessing), PCA(n_components=trial.suggest_float("pca__n_components", 0.90, 0.99))),
        X_train,
        y_train
    ),
    "lightgbm_with_pca": lambda trial: objective_lgbm(
        trial,
        make_pipeline(clone(preprocessing), PCA(n_components=trial.suggest_float("pca__n_components", 0.90, 0.99))),
        X_train,
        y_train
    )
}
pca_model_names = ["ridge_with_pca", "histgradientboosting_with_pca", "xgboost_with_pca", "lightgbm_with_pca"]

for name in pca_model_names:
    print(f"\n{'='*80}\nOptimizing {name.upper()} - 10 trials\n{'='*80}")

    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
    study.optimize(lambda trial: pca_objectives[name](trial), n_trials=10, show_progress_bar=True)

    best_params = study.best_params
    base_name = name.replace("_with_pca", "")

    # Build final PCA pipeline
    final_model = create_pca_pipeline(base_name, best_params)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    test_f1 = f1_score(y_test, y_pred) * 100
    cv_f1 = (1 - study.best_value) * 100

    pca_results[name] = {"pipeline": final_model, "test_f1": test_f1, "cv_f1": cv_f1}

    # Optional: MLflow logging
    with mlflow.start_run(run_name=f"{name}_optuna"):
        mlflow.log_param("model_family", base_name)
        mlflow.log_param("uses_pca", True)
        mlflow.log_params(best_params)
        mlflow.log_metric("cv_F1", cv_f1)
        mlflow.log_metric("test_F1", test_f1)
        signature = infer_signature(X_train, final_model.predict(X_train))
        mlflow.sklearn.log_model(final_model, artifact_path="airline_model_with_pca", signature=signature, input_example=X_train)

print("✓ STEP 6: All PCA models optimized and logged.")

# =============================================================================
# STEP 7: GLOBAL BEST MODEL
# =============================================================================
all_results = {}
all_results.update(results)
all_results.update(pca_results)

global_best_name = max(all_results, key=lambda k: all_results[k]["test_f1"])
global_best_f1 = all_results[global_best_name]["test_f1"]
global_best_cv_f1 = all_results[global_best_name]["cv_f1"]
global_best_pipeline = all_results[global_best_name]["pipeline"]
uses_pca = "with_pca" in global_best_name

print("\n" + "=" * 80)
print("GLOBAL BEST AIRLINE CLASSIFIER")
print("=" * 80)
print(f"Global best model key: {global_best_name}")
print(f"Global best CV F1:    {global_best_cv_f1:.2f}%")
print(f"Global best Test F1:  {global_best_f1:.2f}%")
print(f"Uses PCA:             {uses_pca}")

# =============================================================================
# SAVE ALL MODELS
# =============================================================================
for model_name, info in all_results.items():
    model_path = f"{base_folder}/models/all_models/with_optuna/{model_name}.pkl"
    joblib.dump(info["pipeline"], model_path)

print("✓ All trained models saved.")

# =============================================================================
# SAVE ALL METRICS
# =============================================================================
metrics_records = []
for model_name, info in all_results.items():
    metrics_records.append({
        "model_name": model_name,
        "cv_f1": info["cv_f1"],
        "test_f1": info["test_f1"],
        "uses_pca": "with_pca" in model_name,
        "tuned": "optuna" in model_name
    })
metrics_df = pd.DataFrame(metrics_records)
metrics_path = f"{base_folder}/metrics/with_optuna/all_f1_scores.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"✓ All metrics saved to {metrics_path}")

# =============================================================================
# STEP 8: Save GLOBAL BEST MODEL
# =============================================================================
def save_model(model, filename="global_best_airline_classifier_optuna.pkl"):
    joblib.dump(model, filename)
    print(f"✓ Model saved to {filename}")

save_model(global_best_pipeline, filename=f"{base_folder}/models/global_best_airline_classifier_optuna.pkl")
print("✓ Global best airline classifier saved.")

end_time = time.monotonic()
elapsed_time = end_time - start_time
minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60
print(f"Elapsed time: {minutes} minutes and {seconds:.2f} seconds")

✓ STEP 1: Preprocessing pipeline created.
✓ STEP 2: Stratified split done. Train size: 103589, Test size: 25898


2025/12/17 22:33:01 INFO mlflow.tracking.fluent: Experiment with name 'airline_satisfaction_optuna' does not exist. Creating a new experiment.


✓ STEP 3: MLflow configured.

Optimizing RIDGE (NO PCA) - 10 trials


  0%|          | 0/10 [00:00<?, ?it/s]

ridge (no PCA) CV F1: 84.86% | Test F1: 84.98%


...

🏃 View run xgboost_with_pca_optuna at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/1/runs/f75468efac8f42b7a8d1880848576b90
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/1

Optimizing LIGHTGBM_WITH_PCA - 10 trials


  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 45009, number of negative: 58580
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017412 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 103589, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.434496 -> initscore=-0.263531
[LightGBM] [Info] Start training from score -0.263531


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing

🏃 View run lightgbm_with_pca_optuna at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/1/runs/94f1604ad9074afeb42047492c280d00
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/1
✓ STEP 6: All PCA models optimized and logged.

GLOBAL BEST AIRLINE CLASSIFIER
Global best model key: lightgbm
Global best CV F1:    95.64%
Global best Test F1:  95.63%
Uses PCA:             False
✓ All trained models saved.
✓ All metrics saved to /content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/metrics/with_optuna/all_f1_scores.csv
✓ Model saved to /content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/models/global_best_airline_classifier_optuna.pkl
✓ Global best airline classifier saved.
Elapsed time: 15 minutes and 33.59 seconds
